**Các vấn đề liên quan đến dữ liệu của dirty_cafe_sales.csv:**
- Giá trị không hợp lệ: "ERROR", "UNKNOWN" xuất hiện ở hầu hết các cột
- Missing values: có nhiều ô nulls (Item: 333, Quantity: 138, Price: 179, Total: 173, Payment: 2579, Location: 3265, Date: 159)
- Kiểu dữ liệu sai: Các cột số chứa string ("ERROR", "UNKNOWN")
- Tính toán không nhất quán: Total Spent không luôn bằng Quantity × Price Per Unit
- Ngày tháng không chuẩn: Chứa "ERROR", "UNKNOWN" thay vì định dạng YYYY-MM-DD

**Các phương pháp làm sạch:**
- Đọc file với na_values=["ERROR", "UNKNOWN"]
- Chuyển các cột số sang numeric (coerce errors → NaN)
- Xử lý missing values:
  + Item: Dùng mode hoặc loại bỏ nếu quá nhiều lỗi
  + Quantity và Price Per Unit: Dùng median theo Item (robust hơn mean)
  + Total Spent: Tính lại từ Quantity × Price nếu có thể, hoặc dùng median
  + Payment Method & Location: Dùng mode hoặc "Unknown"
  + Transaction Date: Loại bỏ hoặc điền forward/backward nếu có Transaction ID thứ tự
- Kiểm tra và sửa tính nhất quán Total = Quantity * Price

In [49]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
df = pd.read_csv("D:\DHV301\\dirty_cafe_sales.csv")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [50]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [51]:
# Kiểm tra missing values.
def missing_report(df):
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    return (pd.DataFrame({'count': miss, 'pct': pct})
              .query('count > 0').sort_values('pct', ascending=False))

print(missing_report(df))

                  count    pct
Location           3265  32.65
Payment Method     2579  25.79
Item                333   3.33
Price Per Unit      179   1.79
Total Spent         173   1.73
Transaction Date    159   1.59
Quantity            138   1.38


In [52]:
# Kiểm tra duplicate toàn hàng
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup/len(df)*100:.1f}%)')

Duplicate rows: 0 (0.0%)


In [53]:
# Kiểm tra dtypes
print(df.dtypes)

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object


In [54]:
# đổi giá trị placeholder như "ERROR" và "UNKNOWN" sang " " để xữ lý thống nhất với missing values
columns_to_clean = ['Item', 'Quantity', 'Price Per Unit', 'Total Spent']
for col in columns_to_clean:
    if col in df.columns:
        df[col] = df[col].replace(['ERROR', 'UNKNOWN', 'error', 'unknown'], 
                                  '', regex=False)

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [55]:
# Chuyển đổi dtypes
# cột số ['Quantity', 'Price Per Unit', 'Total Spent'] sang float
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

#cột ngày tháng ['Transaction Date'] sang datetime
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# Kiểm tra sau khi convert
df.dtypes

Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

In [56]:
#  Xử lý Item
valid_items = ['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Sandwich', 'Juice', 'Tea']
df['Item'] = df['Item'].where(df['Item'].isin(valid_items), np.nan)

#  Điền giá trị số theo group (theo Item)
for col in ['Quantity', 'Price Per Unit']:
    df[col] = df.groupby('Item')[col].transform(lambda x: x.fillna(x.median()))

#  Tính lại Total Spent nếu có thể
mask = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price Per Unit']

# Điền còn lại bằng median
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())

#  Xử lý categorical
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

# Điền ngày còn thiếu
df['Transaction Date'] = df['Transaction Date'].ffill()

#  Loại bỏ dòng quá lỗi 
df = df.dropna(subset=['Item', 'Transaction Date'])

In [57]:
df.info()

<class 'pandas.DataFrame'>
Index: 9031 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9031 non-null   str           
 1   Item              9031 non-null   str           
 2   Quantity          9031 non-null   float64       
 3   Price Per Unit    9031 non-null   float64       
 4   Total Spent       9031 non-null   float64       
 5   Payment Method    9031 non-null   str           
 6   Location          9031 non-null   str           
 7   Transaction Date  9031 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 635.0 KB
